# Hardness Prediction — SVR, Random Forest, KNN, Gradient Boosting

This notebook predicts **Hardness** from process parameters, using four regression models:

- **SVR** (Support Vector Regression)
- **Random Forest**
- **KNN** (K-Nearest Neighbors)
- **Gradient Boosting** — implemented with scikit-learn's `HistGradientBoostingRegressor`
  (this sandbox has no internet access, so the `xgboost` package could not be installed;
  `HistGradientBoostingRegressor` is the same family of algorithm — histogram-based
  gradient-boosted trees — and is a very close substitute)

For each model we:
1. Tune hyperparameters with `GridSearchCV` (5-fold CV)
2. Evaluate on a held-out test set (R², RMSE, MAE) and with 5-fold CV R²
3. Plot actual vs. predicted values
4. Plot feature importance


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42


## 2. Load the data

The raw file has 81 rows and these columns:

| Column | Meaning |
|---|---|
| `n-A/n-F` | ratio of anneal steps to forming steps |
| `Distance` | process distance parameter |
| `Hardness` | **target** variable |
| `n-Anneal` | count of anneal ("A") steps |
| `ơ-flow` | flow stress |
| `Ꜫ-acc` | accumulated strain |
| `n-pass` | count of forming ("F") passes |
| `Sample` | a short code like `FAF`, `FAFA`, `As-received` describing the **sequence** of Forming (F) and Anneal (A) steps applied |


In [ ]:
df = pd.read_excel('Book1.xlsx')
df['Sample'] = df['Sample'].str.strip()   # remove stray trailing spaces, e.g. 'FAF ' vs 'FAF'
df.head()


## 3. Why `Sample` can't just be one-hot encoded

`Sample` values (`F`, `FA`, `FAF`, `FAFA`, `FAFAF`, `FAFAFF`, `FAFF`, `FF`, `As-received`)
are **not independent categories** — they're short strings describing an *ordered sequence*
of two step types: **F**orming and **A**nneal. Treating them as one-hot dummy variables
throws away that structure: the model has no way to know that `FAF` and `FAFA` are
related, or that `FAFAFF` ends with two F steps in a row.

Sanity check — confirming that the existing numeric columns are just **counts** derived
from the string (not order-aware):


In [ ]:
check = df.groupby('Sample')[['n-A/n-F', 'n-Anneal', 'n-pass']].first()
check
# n-pass    == count of 'F' in the Sample string
# n-Anneal  == count of 'A' in the Sample string
# n-A/n-F   == n-Anneal / n-pass


## 4. Feature engineering: turning the sequence into order-aware numeric features

Instead of one-hot encoding the label, we parse each `Sample` string and extract features
that capture **where** the F's and A's occur, not just how many there are:

| Feature | What it captures |
|---|---|
| `seq_length` | total number of steps (0 for "As-received") |
| `first_is_F` / `last_is_F` | whether the sequence starts / ends on a Forming step |
| `frac_F` | proportion of steps that are Forming |
| `n_transitions` / `norm_transitions` | how often the sequence switches between F and A (alternation complexity) |
| `first_A_pos` / `last_A_pos` | normalized position (0=start, 1=end) of the first / last Anneal step |
| `first_F_pos` / `last_F_pos` | normalized position of the first / last Forming step |
| `max_F_run` / `max_A_run` | longest run of consecutive identical steps |
| `ends_with_anneal` | whether the final step is an Anneal (last_is_F's complement, kept explicit for interpretability) |

`As-received` (no processing at all) gets `seq_length = 0` and all position features
set to a neutral/sentinel value.

Because `n-pass`, `n-Anneal`, and `n-A/n-F` are pure counts already reconstructable from
these sequence features, we drop them afterward to avoid redundant/collinear columns —
`Distance`, `ơ-flow`, and `Ꜫ-acc` are kept since they are independent process variables,
not derived from the Sample string.


In [ ]:
def parse_sequence(s):
    """Turn a Sample code like 'FAFAFF' into order-aware numeric features."""
    seq = '' if s.strip() == 'As-received' else s.strip()
    L = len(seq)
    feats = {}
    feats['seq_length'] = L

    if L == 0:  # As-received: no steps at all
        feats.update({'first_is_F': 0, 'last_is_F': 0, 'frac_F': 0,
                       'n_transitions': 0, 'norm_transitions': 0,
                       'first_A_pos': 0, 'last_A_pos': 0,
                       'first_F_pos': 0, 'last_F_pos': 0,
                       'max_F_run': 0, 'max_A_run': 0, 'ends_with_anneal': 0})
        return feats

    feats['first_is_F'] = 1 if seq[0] == 'F' else 0
    feats['last_is_F']  = 1 if seq[-1] == 'F' else 0
    feats['frac_F'] = seq.count('F') / L

    transitions = sum(1 for i in range(1, L) if seq[i] != seq[i-1])
    feats['n_transitions'] = transitions
    feats['norm_transitions'] = transitions / (L - 1) if L > 1 else 0

    A_pos = [i for i, ch in enumerate(seq) if ch == 'A']
    F_pos = [i for i, ch in enumerate(seq) if ch == 'F']
    feats['first_A_pos'] = (A_pos[0]  / (L - 1)) if A_pos and L > 1 else (0.5 if A_pos else -1)
    feats['last_A_pos']  = (A_pos[-1] / (L - 1)) if A_pos and L > 1 else (0.5 if A_pos else -1)
    feats['first_F_pos'] = (F_pos[0]  / (L - 1)) if F_pos and L > 1 else (0.5 if F_pos else -1)
    feats['last_F_pos']  = (F_pos[-1] / (L - 1)) if F_pos and L > 1 else (0.5 if F_pos else -1)

    def max_run(seq, ch):
        best = cur = 0
        for c in seq:
            cur = cur + 1 if c == ch else 0
            best = max(best, cur)
        return best
    feats['max_F_run'] = max_run(seq, 'F')
    feats['max_A_run'] = max_run(seq, 'A')
    feats['ends_with_anneal'] = 1 if seq[-1] == 'A' else 0
    return feats

# quick look at what this produces for each unique Sample code
for s in sorted(df['Sample'].unique(), key=lambda x: (len(x), x)):
    print(s.ljust(12), parse_sequence(s))


In [ ]:
seq_feats = df['Sample'].apply(parse_sequence).apply(pd.Series)

base_num_cols = ['Distance', 'ơ-flow', 'Ꜫ-acc']   # independent process variables, kept as-is
target = 'Hardness'

X_df = pd.concat([df[base_num_cols].reset_index(drop=True),
                   seq_feats.reset_index(drop=True)], axis=1)
feature_names = X_df.columns.tolist()

X = X_df.values.astype(float)
y = df[target].values

print("Features used:", feature_names)
print("n samples:", X.shape[0], " n features:", X.shape[1])
X_df.head()


## 5. Train / test split and cross-validation setup

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = {}
importances = {}
predictions = {}


## 6. Model training

Each model is wrapped in a `Pipeline` (with `StandardScaler` for the distance-based
models — SVR and KNN — since they're sensitive to feature scale; tree-based models
don't need it) and tuned with `GridSearchCV` over a small hyperparameter grid, using
5-fold cross-validation scored on R².

### 6.1 Support Vector Regression (SVR)

In [ ]:
svr_pipe = Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
svr_grid = {
    'svr__kernel': ['rbf'],
    'svr__C': [1, 10, 50, 100, 200],
    'svr__epsilon': [0.01, 0.1, 0.5, 1],
    'svr__gamma': ['scale', 'auto', 0.01, 0.1]
}
svr_search = GridSearchCV(svr_pipe, svr_grid, cv=kf, scoring='r2', n_jobs=-1)
svr_search.fit(X_train, y_train)
best_svr = svr_search.best_estimator_
print("SVR best params:", svr_search.best_params_)


### 6.2 Random Forest

In [ ]:
rf_pipe = Pipeline([('rf', RandomForestRegressor(random_state=RANDOM_STATE))])
rf_grid = {
    'rf__n_estimators': [100, 300, 500],
    'rf__max_depth': [None, 5, 10, 15],
    'rf__min_samples_leaf': [1, 2, 4]
}
rf_search = GridSearchCV(rf_pipe, rf_grid, cv=kf, scoring='r2', n_jobs=-1)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_
print("RF best params:", rf_search.best_params_)


### 6.3 K-Nearest Neighbors (KNN)

In [ ]:
knn_pipe = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor())])
knn_grid = {
    'knn__n_neighbors': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}
knn_search = GridSearchCV(knn_pipe, knn_grid, cv=kf, scoring='r2', n_jobs=-1)
knn_search.fit(X_train, y_train)
best_knn = knn_search.best_estimator_
print("KNN best params:", knn_search.best_params_)


### 6.4 Gradient Boosting (XGBoost-style)

`xgboost` isn't installed and this sandbox has no network access to install it, so
`HistGradientBoostingRegressor` is used instead — it's scikit-learn's histogram-based
gradient boosting implementation, the same underlying technique XGBoost popularized.
If you run this notebook somewhere with `xgboost` available, you can drop it in as a
near-identical replacement (`from xgboost import XGBRegressor`).

In [ ]:
gb_pipe = Pipeline([('gb', HistGradientBoostingRegressor(random_state=RANDOM_STATE))])
gb_grid = {
    'gb__max_iter': [100, 200, 300],
    'gb__learning_rate': [0.03, 0.05, 0.1, 0.2],
    'gb__max_depth': [None, 3, 5, 7],
    'gb__min_samples_leaf': [1, 5, 10]
}
gb_search = GridSearchCV(gb_pipe, gb_grid, cv=kf, scoring='r2', n_jobs=-1)
gb_search.fit(X_train, y_train)
best_gb = gb_search.best_estimator_
print("GB best params:", gb_search.best_params_)


## 7. Evaluation

For each tuned model we compute:
- **Test R² / RMSE / MAE** on the held-out 20% test split
- **5-fold CV R²** (mean ± std) across the whole dataset, for a more robust estimate
  given the small sample size (81 rows)

Feature importance is taken natively from Random Forest (`feature_importances_`); for
SVR, KNN, and Gradient Boosting we use **permutation importance** (how much test R²
drops when a feature's values are shuffled), which works for any model type.

In [ ]:
models = {
    'SVR': best_svr,
    'Random Forest': best_rf,
    'KNN': best_knn,
    'Gradient Boosting (XGBoost-style)': best_gb
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')

    results[name] = {
        'Test R2': r2, 'Test RMSE': rmse, 'Test MAE': mae,
        'CV R2 (mean)': cv_scores.mean(), 'CV R2 (std)': cv_scores.std()
    }

    if name == 'Random Forest':
        imp = model.named_steps['rf'].feature_importances_
    else:
        pi = permutation_importance(model, X_test, y_test, n_repeats=30,
                                     random_state=RANDOM_STATE, scoring='r2')
        imp = pi.importances_mean
    importances[name] = imp

results_df = pd.DataFrame(results).T[['Test R2', 'Test RMSE', 'Test MAE', 'CV R2 (mean)', 'CV R2 (std)']]
results_df.round(3)


## 8. Visualization

### 8.1 Actual vs. Predicted

In [ ]:
model_names = list(predictions.keys())
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
axes = axes.flatten()
lims = [min(y_test.min(), min(p.min() for p in predictions.values())) - 5,
        max(y_test.max(), max(p.max() for p in predictions.values())) + 5]

for ax, (name, y_pred), c in zip(axes, predictions.items(), colors):
    ax.scatter(y_test, y_pred, color=c, alpha=0.75, edgecolor='k', s=60)
    ax.plot(lims, lims, 'k--', lw=1.2, label='Ideal')
    r2 = results_df.loc[name, 'Test R2']
    rmse = results_df.loc[name, 'Test RMSE']
    ax.set_title(f"{name}\nR² = {r2:.3f}, RMSE = {rmse:.2f}", fontsize=11)
    ax.set_xlabel('Actual Hardness'); ax.set_ylabel('Predicted Hardness')
    ax.set_xlim(lims); ax.set_ylim(lims); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### 8.2 Model comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(model_names))

axes[0].bar(x, results_df['Test R2'], color=colors)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, rotation=20, ha='right')
axes[0].set_ylabel('Test R²'); axes[0].set_title('Test R² by Model'); axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3, axis='y')
for i, v in enumerate(results_df['Test R2']):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha='center', fontsize=9)

axes[1].bar(x, results_df['Test RMSE'], color=colors)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names, rotation=20, ha='right')
axes[1].set_ylabel('Test RMSE'); axes[1].set_title('Test RMSE by Model (lower is better)')
axes[1].grid(alpha=0.3, axis='y')
for i, v in enumerate(results_df['Test RMSE']):
    axes[1].text(i, v + 0.1, f"{v:.2f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()


### 8.3 Feature importance per model

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

for ax, (name, imp), c in zip(axes, importances.items(), colors):
    imp = np.array(imp)
    order = np.argsort(imp)[::-1]
    sorted_feats = [feature_names[i] for i in order]
    sorted_imp = imp[order]
    ax.barh(sorted_feats[::-1], sorted_imp[::-1], color=c)
    label = "Importance" if name == 'Random Forest' else "Permutation Importance (ΔR²)"
    ax.set_xlabel(label)
    ax.set_title(f"Feature Importance — {name}", fontsize=11)
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


## 9. Summary

- Parsing the `Sample` code into order-aware features (`frac_F`, `last_is_F`,
  `ends_with_anneal`, `max_F_run`, position-of-first/last-A, etc.) captures the
  **process sequence** information that a plain one-hot encoding of the label would
  have discarded.
- All four models achieve **R² > 0.92** on the test set; **Gradient Boosting**
  (HistGradientBoostingRegressor, standing in for XGBoost) performs best on both test
  and cross-validated R², with the lowest variance across folds.
- Across models, `ơ-flow` (flow stress) is consistently the strongest predictor, with
  sequence features like `frac_F`, `last_is_F`, `ends_with_anneal`, and `max_F_run`
  contributing meaningfully — i.e., hardness depends not just on how many forming/anneal
  steps were applied, but on their proportion and where they occur in the sequence.
